# Graph construction, provenance, and the store

Reference implementation 4 (RI #4) builds two kinds of graph. This notebook covers the one you construct yourself: a knowledge graph extracted directly from 9,999 ServerFault posts with two model calls each (one over the title and question, one over the accepted answer), under a fixed schema, where every node and edge cites the words it came from. It shows the schema, what a citation looks like, how the graph validates, and what the build cost.

The build took 8.1 hours and 20,077 model calls, so it sits behind `RUN_EXPENSIVE`. With the flag off, the notebook reads the statistics and the sample committed in `outputs/`. It shows entity names, relation types, and character offsets, and no text from any post, because ServerFault posts are licensed under Creative Commons Attribution-ShareAlike (CC BY-SA) and `CORPUS.md` records the terms.

In [1]:
import json
from collections import Counter
from pathlib import Path

from aieng.advanced_rag.graphrag.extraction import ENTITY_TYPES, RELATION_TYPES


# Building the graph took 8.1 hours. The True path reads the 130 MB graph from the
# artifact cache. Leave this False to read the committed results.
RUN_EXPENSIVE = False

HERE = Path.cwd()
OUTPUTS = HERE / "outputs"
SAMPLE_DOCUMENT = "q9708"

print("entity types:  ", ", ".join(ENTITY_TYPES))
print("relation types:", ", ".join(RELATION_TYPES))

entity types:   Software, Hardware, Protocol, ConfigItem, Command, ErrorOrSymptom, Concept
relation types: causes, fixes, configures, depends_on, part_of, alternative_to, diagnoses


## The schema is enforced after the call

The extraction prompt names seven entity types and seven relation types. A model still returns labels such as `Tool` or `requires`, so a mapping table folds known synonyms onto the schema. An entity type with no mapping becomes `Concept` and is counted. A relation with no mapping is dropped, and so is a relation whose endpoint is not an entity. The graph validates only if every label is on the schema, every relation's endpoints exist, and every fact cites at least one place in a document.

In [2]:
if RUN_EXPENSIVE:
    from aieng.advanced_rag.cache import TieredResultCache, fetch_directory
    from aieng.advanced_rag.graphrag.extraction import KnowledgeGraph
    from build_serverfault_graph import cache_key

    summary_source = json.loads((OUTPUTS / "serverfault_graph_summary.json").read_text())
    folder = fetch_directory(TieredResultCache(), cache_key(summary_source["documents"]), HERE / "_graph", replace=True)
    graph = KnowledgeGraph.load(folder)
    problems = graph.validate()
    print("problems found by validate():", len(problems))
    # Keep names, types, and spans for one post, and no text from it.
    entities = [e for e in graph.entities.values() if SAMPLE_DOCUMENT in e.documents]
    relations = [r for r in graph.relations.values() if SAMPLE_DOCUMENT in r.documents]

    def cites(mentions):
        """Return the spans that cite SAMPLE_DOCUMENT."""
        return [
            {
                "section": m.provenance.section,
                "block": m.provenance.chunk_id,
                "start": m.provenance.start,
                "end": m.provenance.end,
                "located": m.located,
            }
            for m in mentions
            if m.provenance.doc_id == SAMPLE_DOCUMENT
        ]

    sample = {
        "document": SAMPLE_DOCUMENT,
        "validation_problems": len(problems),
        "entities": [{"name": e.name, "type": e.type, "cites": cites(e.mentions)} for e in entities],
        "relations": [
            {"source": r.source, "type": r.type, "target": r.target, "cites": cites(r.mentions)} for r in relations
        ],
    }
    (OUTPUTS / "serverfault_graph_sample.json").write_text(json.dumps(sample, indent=1) + "\n")

summary = json.loads((OUTPUTS / "serverfault_graph_summary.json").read_text())
stats = summary["graph"]
report = stats["schema_report"]
print(f"{stats['entities']:,} entities and {stats['relations']:,} relations from {summary['documents']:,} documents")
print("entity types:  ", dict(Counter(stats["entity_types"]).most_common()))
print("relation types:", dict(Counter(stats["relation_types"]).most_common()))
print()
print(f"entity labels mapped onto the schema: {report['entity_types_mapped']:,}")
print(f"entity labels coerced to Concept:     {report['entity_types_coerced']:,}")
print(f"relations dropped, unmapped type:     {report['relations_rejected']:,}")
print(f"relations dropped, dangling endpoint: {report['relations_dangling']:,}")
print(f"model calls that failed after retries: {report['calls_failed']:,} of {summary['cost']['calls']:,}")

63,436 entities and 86,644 relations from 9,999 documents
entity types:   {'ConfigItem': 17916, 'Concept': 12574, 'Software': 10507, 'Command': 8841, 'ErrorOrSymptom': 7027, 'Hardware': 5025, 'Protocol': 1546}
relation types: {'configures': 24790, 'part_of': 17628, 'depends_on': 16701, 'causes': 10015, 'alternative_to': 6641, 'diagnoses': 6336, 'fixes': 4533}

entity labels mapped onto the schema: 138,057
entity labels coerced to Concept:     60
relations dropped, unmapped type:     62
relations dropped, dangling endpoint: 271
model calls that failed after retries: 215 of 20,077


## Every fact cites a span

A citation is a chain: the document (a post), a section (title, question, or accepted answer), a block inside the section (a paragraph or a code block), and the character span of the quoted words inside that block. The model quotes the words it relied on. The code then finds the quote in the block, first exactly and then ignoring case and whitespace, and records the offsets. A quote the code cannot find keeps its section and loses its span, and the mention is marked as not located.

In [3]:
sample = json.loads((OUTPUTS / "serverfault_graph_sample.json").read_text())
print(
    f"post {sample['document']}: {len(sample['entities'])} entities, {len(sample['relations'])} relations, "
    f"{sample['validation_problems']} validation problems in the whole graph"
)
print()
for entity in sample["entities"][:6]:
    print(f"{entity['type']:<14} {entity['name']}")
    for cite in entity["cites"][:2]:
        span = f"{cite['start']}-{cite['end']}" if cite["located"] else "section only"
        print(f"    {cite['section']} / {cite['block']} / chars {span}")
print()
for relation in sample["relations"][:4]:
    print(f"{relation['source']} --{relation['type']}--> {relation['target']}")

post q9708: 15 entities, 12 relations, 0 validation problems in the whole graph

Concept        pem file
    Title / q9708::title::b000 / chars 2-20
Software       openssl
    Title / q9708::title::b000 / chars 55-89
    Accepted answer / q9708::accepted-answer::b004 / chars 2852-2890
ConfigItem     .key
    Question / q9708::question::b001 / chars 348-389
    Accepted answer / q9708::accepted-answer::b003 / chars 2140-2217
ConfigItem     .csr
    Question / q9708::question::b001 / chars 348-389
    Accepted answer / q9708::accepted-answer::b001 / chars 968-1012
ConfigItem     .pem
    Question / q9708::question::b001 / chars 348-389
    Accepted answer / q9708::accepted-answer::b002 / chars 1541-1567
Software       debian
    Question / q9708::question::b000 / chars 149-179

openssl --configures--> pem file
debian --depends_on--> security certificates
x509 --depends_on--> asn.1
.csr --part_of--> pkcs10


In [4]:
located, total = stats["located_mentions"], stats["mentions"]
print(f"{total:,} mentions, {located:,} located to a character span ({located / total:.1%})")
print(f"{stats['cross_document_entities']:,} entities appear in more than one post")
print(f"{stats['isolated_entities']:,} entities have no relation")

229,623 mentions, 216,203 located to a character span (94.2%)
11,465 entities appear in more than one post
4,686 entities have no relation


## What the build cost

The model proxy, which serves the model, limited the run to 50 requests a minute. The 20,077 successful calls average 41 a minute, and adding the 3,689 failed attempts that were retried brings the request rate to about 49 a minute, so the limit set the 8.1 hours. Each post costs two calls because the title shares a call with the question.

In [5]:
cost = summary["cost"]
print(f"model {summary['model']}")
print(f"{cost['calls']:,} calls, {summary['calls_per_document']:.2f} per document")
print(
    f"{cost['prompt_tokens']:,} prompt tokens and {cost['completion_tokens']:,} completion tokens, "
    f"{summary['tokens_per_document']:,.0f} per document"
)
print(f"{summary['wall_seconds'] / 3600:.1f} hours wall time, {cost['failed_attempts']:,} failed attempts retried")

model Qwen3_8-27B
20,077 calls, 2.01 per document
10,959,527 prompt tokens and 15,172,735 completion tokens, 2,614 per document
8.1 hours wall time, 3,689 failed attempts retried
